# Marmousi2 Acoustic bv1.2 Validation

This notebook runs the same staged validation entry point as the Python script in this folder. It does not modify the original Marmousi2 example notebooks.

## Stages

- `check`: read model/survey/observed data and verify backend setup.
- `forward`: run one true-model single-shot forward check.
- `inversion10`: run a short synthetic-true inversion.
- `inversion100`: run a longer sanity inversion when needed.
- `all`: run `check`, `forward`, and `inversion10`.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "ADFWI").exists():
    REPO_ROOT = Path("/liufeng1afs/project/04_Inversion/ADFWI-github")

SCRIPT = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12" / "run_validation.py"
OUTPUT_ROOT = REPO_ROOT / "examples" / "validation" / "marmousi2_acoustic_bv12" / "outputs"
SCRIPT

In [ ]:
def run_stage(stage: str, *, dry_run: bool = True, overwrite: bool = False, device: str = "npu:0", extra_args: list[str] | None = None):
    command = [sys.executable, str(SCRIPT), stage, "--device", device, "--output-root", str(OUTPUT_ROOT)]
    if dry_run:
        command.append("--dry-run")
    if overwrite:
        command.append("--overwrite")
    if extra_args:
        command.extend(extra_args)
    proc = subprocess.run(command, cwd=str(REPO_ROOT), text=True, capture_output=True, check=False)
    if proc.stderr:
        print(proc.stderr)
    if proc.stdout:
        print(proc.stdout)
    if proc.returncode != 0:
        raise RuntimeError(f"stage {stage} failed with return code {proc.returncode}")
    return json.loads(proc.stdout[proc.stdout.find("{"):])


## Preview Commands

Run this cell first. It prints the exact commands that will be executed without running the expensive validation.

In [ ]:
plan = run_stage("all", dry_run=True, device="npu:0")
[(stage["stage"], stage["output_dir"]) for stage in plan["stages"]]

## Run Lightweight Stages

Set `RUN_VALIDATION = True` to execute the staged check. The default is safe for opening and running the notebook top to bottom.

In [ ]:
RUN_VALIDATION = False

if RUN_VALIDATION:
    result = run_stage("all", dry_run=False, overwrite=True, device="npu:0")
else:
    result = {"status": "skipped", "reason": "set RUN_VALIDATION=True to run check/forward/inversion10"}
result

## Optional 100-Iteration Run

Run this only after the 10-iteration stage looks reasonable.

In [ ]:
RUN_100_ITER = False

if RUN_100_ITER:
    long_result = run_stage("inversion100", dry_run=False, overwrite=True, device="npu:0")
else:
    long_result = {"status": "skipped", "reason": "set RUN_100_ITER=True to run inversion100"}
long_result

## Inspect Outputs

After running a stage, inspect `summary.json`, `loss_history.csv`, `loss_curve.png`, and `vp_initial_final_delta.png` under the output directory reported above.